# Exercice : Affiner un modèle sur son propre corpus

Dans cet exercice, nous reprenons le principe du notebook de démonstration sur l'affinage, mais en l'ajustant à votre corpus personnel. 

Voici un aperçu de la chaîne de traitement
```text
corpus personnel
→ inspection
→ formatage
→ tokenisation
→ affinage
→ génération / comparaison
```

## 1. Installer et importer les bibliothèques

Nous allons utiliser :

- `pandas` pour charger et inspecter le corpus ;
- `datasets` pour créer un dataset compatible avec Hugging Face ;
- `transformers` pour charger le modèle, le tokenizer et le `Trainer` ;
- `torch` pour l'exécution CPU/GPU.

In [ ]:
# Dans un environnement neuf, décommenter au besoin :
# !pip install transformers datasets torch pandas

from pathlib import Path
import json
import re

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

## 2. Configuration générale

Vous pouvez modifier :

- `CORPUS_PATH` pour pointer vers votre fichier ;
- `MODEL_NAME` pour choisir un modèle ;
- `FORMAT_MODE` pour choisir la manière de transformer le corpus ;
- `RUN_TRAINING` pour lancer ou non l'entraînement.

Par défaut, l'entraînement est désactivé pour éviter de lancer un calcul long par accident.

In [ ]:
# Modèle de départ.
# "distilgpt2" est plus léger que "gpt2", donc plus pratique pour un atelier.
MODEL_NAME = "distilgpt2"

# Chemin vers le corpus personnel.
# Formats acceptés dans ce notebook : .txt, .csv, .jsonl
# Exemple :
# CORPUS_PATH = Path("./mon_corpus.csv")
# CORPUS_PATH = Path("./formatted_epigrams_eng_genre.txt")
CORPUS_PATH = None

# Si aucun fichier n'est fourni, on utilise un mini-corpus de secours.
USE_DEMO_CORPUS_IF_MISSING = True

# Mode de formatage :
# - "auto" : essaie de deviner selon les colonnes disponibles
# - "plain_text" : utilise seulement le texte
# - "tagged" : crée un format balisé avec auteur/genre/texte
# - "instruction" : crée un format instruction / input / output
FORMAT_MODE = "auto"

# Colonnes possibles.
# Le notebook essaie aussi quelques variantes fréquentes.
TEXT_COLUMN = "text"
AUTHOR_COLUMN = "author"
GENRE_COLUMN = "genre"
INSTRUCTION_COLUMN = "instruction"
INPUT_COLUMN = "input"
OUTPUT_COLUMN = "output"

# Paramètres d'entraînement.
OUTPUT_DIR = Path("./checkpoints/exercice_affinage")
FINAL_MODEL_DIR = Path("./modele_affine_exercice_final")

RUN_TRAINING = False   # Mettre True pour lancer réellement l'entraînement.
MAX_STEPS = 30         # Petit nombre pour une démo ou un exercice.
MAX_LENGTH = 192       # Longueur maximale des séquences tokenisées.
VALIDATION_RATIO = 0.2
SEED = 42

# Prompt utilisé plus tard pour tester la génération.
PROMPT_TEMPLATE = """<author>Anonymous</author>
<genre>funerary</genre>
<text>"""

## 3. Mini-corpus de secours

Si aucun fichier n'est fourni, le notebook utilise ce petit corpus.

Ce n'est pas un bon dataset pour entraîner un vrai modèle.  
C'est simplement un filet de sécurité pour que le notebook puisse être exécuté de bout en bout.

Dans un vrai exercice, remplacez ce mini-corpus par vos propres données.

In [ ]:
DEMO_EXAMPLES = [
    {
        "author": "Anonymous",
        "genre": "funerary",
        "text": "A small stone keeps the name of a child; the sea keeps the rest of his story."
    },
    {
        "author": "Anonymous",
        "genre": "funerary",
        "text": "Do not pass too quickly, stranger: this dust once loved the morning light."
    },
    {
        "author": "Anyte",
        "genre": "animal",
        "text": "Here lies the cicada, singer of noon, silent now beneath a little leaf."
    },
    {
        "author": "Anonymous",
        "genre": "erotic",
        "text": "Love wrote her name in wine; morning erased it, but not from my heart."
    },
    {
        "author": "Anonymous",
        "genre": "votive",
        "text": "To the goddess, I dedicate this net, these hooks, and the memory of a lucky sea."
    },
    {
        "author": "Meleager",
        "genre": "erotic",
        "text": "The garland fades, but desire keeps its own spring."
    },
    {
        "author": "Anonymous",
        "genre": "satirical",
        "text": "He promised wisdom to all; then charged admission at the door."
    },
    {
        "author": "Anonymous",
        "genre": "funerary",
        "text": "Beneath this marker sleeps a dog who guarded the house better than its master."
    },
]

demo_df = pd.DataFrame(DEMO_EXAMPLES)
demo_df

## 4. Charger un corpus personnel

Cette fonction accepte trois types de fichiers.

### Fichier `.txt`

Deux possibilités :

1. Le fichier contient déjà des exemples séparés par `<end>`.
2. Le fichier contient des paragraphes séparés par des lignes vides.

### Fichier `.csv`

Le fichier doit contenir au moins une colonne textuelle.

Exemples de noms acceptés automatiquement :

- `text`
- `texte`
- `text_fr`
- `content`
- `epigram`
- `epigramme`

### Fichier `.jsonl`

Chaque ligne doit être un objet JSON.

Exemple :

```json
{"instruction": "Classe cette épigramme.", "input": "...", "output": "funéraire"}
```

In [ ]:
def load_txt_corpus(path):
    content = path.read_text(encoding="utf-8").strip()

    if "<end>" in content:
        parts = [part.strip() for part in content.split("<end>") if part.strip()]
        texts = [part + "\n<end>" for part in parts]
    else:
        # Séparation simple par paragraphes.
        parts = re.split(r"\n\s*\n", content)
        texts = [part.strip() for part in parts if part.strip()]

    return pd.DataFrame({"text": texts})


def load_jsonl_corpus(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


def load_corpus(path=None):
    if path is None:
        if USE_DEMO_CORPUS_IF_MISSING:
            print("Aucun fichier fourni : utilisation du mini-corpus de secours.")
            return demo_df.copy()
        raise ValueError("Aucun fichier fourni. Définissez CORPUS_PATH.")

    path = Path(path)

    if not path.exists():
        if USE_DEMO_CORPUS_IF_MISSING:
            print(f"Fichier introuvable : {path}")
            print("Utilisation du mini-corpus de secours.")
            return demo_df.copy()
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    suffix = path.suffix.lower()

    if suffix == ".txt":
        return load_txt_corpus(path)

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".jsonl":
        return load_jsonl_corpus(path)

    raise ValueError(
        f"Format non reconnu : {suffix}. "
        "Formats acceptés : .txt, .csv, .jsonl"
    )


raw_df = load_corpus(CORPUS_PATH)
raw_df.head()

## 5. Inspecter les données

Avant d'affiner un modèle, on inspecte toujours le corpus.

Questions simples :

- Combien d'exemples avons-nous ?
- Quelles colonnes sont disponibles ?
- Quelle colonne contient le texte ?
- Les lignes sont-elles vides ou très courtes ?
- Les labels ou métadonnées sont-ils cohérents ?

In [ ]:
print("Dimensions du corpus :", raw_df.shape)
print("Colonnes :", list(raw_df.columns))

display(raw_df.head())

missing_by_column = raw_df.isna().sum().sort_values(ascending=False)
missing_by_column

## 6. Trouver automatiquement les colonnes utiles

Pour rendre le notebook plus flexible, on cherche les colonnes à partir de plusieurs noms possibles.

Cela permet de charger des corpus différents sans tout réécrire.

Ce code n'est pas en lien avec la formation. Vous pouvez passez à la prochaine étape. 

In [ ]:
def find_column(df, candidates):
    """Retourne la première colonne existante parmi une liste de candidates."""
    for col in candidates:
        if col in df.columns:
            return col
    return None


TEXT_CANDIDATES = [
    TEXT_COLUMN,
    "text",
    "texte",
    "text_fr",
    "content",
    "contenu",
    "epigram",
    "epigramme",
    "passage",
]

AUTHOR_CANDIDATES = [
    AUTHOR_COLUMN,
    "author",
    "auteur",
    "author_fr",
    "attribution",
]

GENRE_CANDIDATES = [
    GENRE_COLUMN,
    "genre",
    "category",
    "categorie",
    "catégorie",
    "label",
]

INSTRUCTION_CANDIDATES = [
    INSTRUCTION_COLUMN,
    "instruction",
    "consigne",
    "prompt",
]

INPUT_CANDIDATES = [
    INPUT_COLUMN,
    "input",
    "entrée",
    "entree",
    "source",
]

OUTPUT_CANDIDATES = [
    OUTPUT_COLUMN,
    "output",
    "sortie",
    "response",
    "réponse",
    "reponse",
    "target",
    "label",
    "type"
]

resolved_columns = {
    "text": find_column(raw_df, TEXT_CANDIDATES),
    "author": find_column(raw_df, AUTHOR_CANDIDATES),
    "genre": find_column(raw_df, GENRE_CANDIDATES),
    "instruction": find_column(raw_df, INSTRUCTION_CANDIDATES),
    "input": find_column(raw_df, INPUT_CANDIDATES),
    "output": find_column(raw_df, OUTPUT_CANDIDATES),
}

resolved_columns

## 7. Nettoyer légèrement le corpus

On retire les lignes entièrement vides et on crée une version propre du tableau.

N'hésitez pas à appliquer les techniques apprises pendant la formation.

In [ ]:
df = raw_df.copy()

# Suppression des lignes entièrement vides.
df = df.dropna(how="all").reset_index(drop=True)

# Si une colonne textuelle existe, on enlève les textes vides.
text_col = resolved_columns["text"]
if text_col is not None:
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 0].reset_index(drop=True)

## VOS PROPRES FONCTIONS ICI

print("Nombre d'exemples après nettoyage :", len(df))
display(df.head())

## 8. Choisir le format d'entraînement

Un modèle génératif apprend à continuer du texte.

Il faut donc transformer nos données en chaînes de caractères qui montrent clairement le comportement attendu.

Nous allons gérer trois formats.

### Format texte simple

```text
Texte du corpus
<end>
```

### Format balisé

```text
<author>Anonymous</author>
<genre>funerary</genre>
<epigram>
Texte du corpus
<end>
```

### Format instruction

```text
### Instruction:
Classe cette épigramme.

### Input:
Texte de l'épigramme

### Response:
funéraire
<end>
```

Le code en lui-même n'est pas en lien avec la formation, mais les principes sont importants.

In [ ]:
def normalize_value(value, default="Unknown"):
    if pd.isna(value):
        return default
    value = str(value).strip()
    return value if value else default


def detect_format_mode(df, resolved_columns):
    """Détermine un mode de formatage raisonnable si FORMAT_MODE vaut auto."""
    if FORMAT_MODE != "auto":
        return FORMAT_MODE

    has_instruction = resolved_columns["instruction"] is not None
    has_output = resolved_columns["output"] is not None
    has_text = resolved_columns["text"] is not None
    has_author_or_genre = (
        resolved_columns["author"] is not None
        or resolved_columns["genre"] is not None
    )

    if has_instruction and has_output:
        return "instruction"

    if has_text and has_author_or_genre:
        return "tagged"

    if has_text:
        return "plain_text"

    raise ValueError(
        "Impossible de déterminer le format. "
        "Ajoutez une colonne de texte ou des colonnes instruction/output."
    )


ACTIVE_FORMAT_MODE = detect_format_mode(df, resolved_columns)
print("Mode de formatage utilisé :", ACTIVE_FORMAT_MODE)

## 9. Transformer chaque ligne en exemple d'entraînement

À vous de formater vos exemples selon le type choisi. 

In [ ]:
def format_training_example(row, mode=ACTIVE_FORMAT_MODE):
    text_col = resolved_columns["text"]
    author_col = resolved_columns["author"]
    genre_col = resolved_columns["genre"]
    instruction_col = resolved_columns["instruction"]
    input_col = resolved_columns["input"]
    output_col = resolved_columns["output"]

    if mode == "plain_text":
        # EXERCICE
        # Récupérez le texte de la ligne avec normalize_value(...)
        # puis retournez une chaîne qui se termine par "\n<end>".
        #
        # Indice :
        # text = normalize_value(row[text_col], default="")
        # return ...
        raise NotImplementedError("Complétez le format plain_text.")

    if mode == "tagged":
        # EXERCICE
        # Construisez un exemple balisé avec (par exemple) :
        # - <author>...</author>
        # - <genre>...</genre>
        # - <epigram>
        # - le texte
        # - <end>
        #
        # Indice :
        # author = normalize_value(row[author_col], default="Unknown") if author_col else "Unknown"
        # genre = normalize_value(row[genre_col], default="unknown") if genre_col else "unknown"
        # text = normalize_value(row[text_col], default="")
        raise NotImplementedError("Complétez le format tagged.")

    if mode == "instruction":
        # EXERCICE
        # Construisez un exemple de type instruction / input / response.
        #
        # Indice :
        # instruction = normalize_value(...)
        # input_text = normalize_value(...)
        # output_text = normalize_value(...)
        raise NotImplementedError("Complétez le format instruction.")

    raise ValueError(f"Mode inconnu : {mode}")


# Cette ligne doit fonctionner une fois la fonction complétée.
df["training_text"] = df.apply(format_training_example, axis=1)
df["training_length_chars"] = df["training_text"].str.len()

display(df[["training_text", "training_length_chars"]].head())

## 10. Inspecter les exemples formatés

Avant d'entraîner, on lit quelques exemples.

In [ ]:
for i, text in enumerate(df["training_text"].head(3), start=1):
    print("=" * 80)
    print(f"EXEMPLE {i}")
    print("=" * 80)
    print(text)
    print()

## 11. Créer un Dataset Hugging Face

Le `Trainer` de Hugging Face travaille avec un objet `Dataset`.

In [ ]:
dataset = Dataset.from_pandas(df[["training_text"]], preserve_index=False)
dataset

## 12. Séparer entraînement et validation

Même dans un exercice, il faut garder l'idée d'une séparation entre les données vues pendant l'entraînement et les données utilisées pour l'évaluation.

In [ ]:
# EXERCICE
# Séparez le dataset en données d'entraînement et données de validation.
#
# Objectif :
# - si le corpus contient au moins 5 exemples, utilisez dataset.train_test_split(...)
# - sinon, réutilisez le même petit dataset pour la démonstration.
#
# Indices :
# split_dataset = dataset.train_test_split(test_size=VALIDATION_RATIO, seed=SEED)
# train_dataset_raw = split_dataset["train"]
# validation_dataset_raw = split_dataset["test"]

if len(dataset) >= 5:
    # À compléter
    raise NotImplementedError("Complétez la séparation train/validation.")
else:
    print("Corpus très petit : on réutilise le même corpus pour une validation de démonstration.")
    train_dataset_raw = dataset
    validation_dataset_raw = dataset

print("Exemples d'entraînement :", len(train_dataset_raw))
print("Exemples de validation :", len(validation_dataset_raw))

## 13. Charger le tokenizer et le modèle

Nous chargeons maintenant le modèle de départ.

GPT-2 et DistilGPT-2 n'ont pas de token de padding par défaut.  
On réutilise donc le token de fin de texte comme token de padding.

In [ ]:
# EXERCICE
# Chargez le tokenizer et le modèle causal à partir de MODEL_NAME.
#
# Indices :
# tokenizer = AutoTokenizer.from_pretrained(...)
# model = AutoModelForCausalLM.from_pretrained(...)
#
# Attention :
# GPT-2 / DistilGPT-2 n'ont pas toujours de token de padding.
# Si tokenizer.pad_token est None, utilisez tokenizer.eos_token.

tokenizer = None
model = None

# À compléter ici

if tokenizer is None or model is None:
    raise NotImplementedError("Chargez le tokenizer et le modèle.")

print("Modèle chargé :", MODEL_NAME)
print("Taille du vocabulaire :", tokenizer.vocab_size)
print("Token de padding :", tokenizer.pad_token)

## 14. Observer la tokenisation d'un exemple

Le modèle ne reçoit pas le texte directement.

Le tokenizer transforme chaque exemple en identifiants numériques.

In [ ]:
example_text = df["training_text"].iloc[0]
encoded_example = tokenizer(example_text, truncation=True, max_length=MAX_LENGTH)
tokens = tokenizer.convert_ids_to_tokens(encoded_example["input_ids"])

pd.DataFrame({
    "position": range(len(tokens)),
    "token": tokens,
    "id": encoded_example["input_ids"],
}).head(100)

## 15. Tokeniser tout le corpus

Cette cellule transforme tous les exemples en séquences numériques.

In [ ]:
def tokenize_function(batch):
    # EXERCICE
    # Tokenisez batch["training_text"] avec :
    # - truncation=True
    # - max_length=MAX_LENGTH
    raise NotImplementedError("Complétez la fonction de tokenisation.")


train_dataset = train_dataset_raw.map(
    tokenize_function,
    batched=True,
    remove_columns=["training_text"],
)

validation_dataset = validation_dataset_raw.map(
    tokenize_function,
    batched=True,
    remove_columns=["training_text"],
)

print(train_dataset)
print(validation_dataset)

## 16. Préparer les lots d'entraînement

Le `DataCollatorForLanguageModeling` crée les lots envoyés au modèle.

Ici, `mlm=False`, parce que GPT-2 est un modèle de langage causal : il apprend à prédire le token suivant.

Ce n'est pas le même objectif que BERT, qui utilise des tokens masqués pendant son pré-entraînement.

In [ ]:
# EXERCICE
# Créez le DataCollatorForLanguageModeling.
#
# Pour GPT-2 / DistilGPT-2, on veut un modèle causal :
# mlm=False

data_collator = None

# À compléter ici

if data_collator is None:
    raise NotImplementedError("Créez le data_collator.")

print("Collator prêt.")

## 17. Configurer l'entraînement

Les paramètres importants :

- `max_steps` : nombre d'étapes d'entraînement ;
- `learning_rate` : taille des ajustements ;
- `batch_size` : nombre d'exemples traités à la fois ;
- `gradient_accumulation_steps` : accumule les gradients sur plusieurs petits lots ;
- `fp16` : accélère parfois l'entraînement sur GPU compatible.

In [ ]:
batch_size = 4 if torch.cuda.is_available() else 2

# EXERCICE
# Complétez les TrainingArguments.
#
# Gardez une configuration courte pour l'atelier :
# - max_steps=MAX_STEPS
# - learning_rate=5e-5
# - logging_steps=5
# - fp16=torch.cuda.is_available()
#
# Ensuite, créez le Trainer avec :
# - model=model
# - args=training_args
# - train_dataset=train_dataset
# - eval_dataset=validation_dataset
# - data_collator=data_collator

training_args = None

# À compléter ici

if training_args is None:
    raise NotImplementedError("Complétez TrainingArguments.")

trainer = None

# À compléter ici

if trainer is None:
    raise NotImplementedError("Créez le Trainer.")

print("Configuration prête.")
print("Batch size :", batch_size)
print("Entraînement activé :", RUN_TRAINING)

## 18. Lancer ou simuler l'entraînement

Par défaut, `RUN_TRAINING = False`.

Pour lancer réellement l'entraînement, retournez à la cellule de configuration et mettez :

```python
RUN_TRAINING = True
```

Ensuite, réexécutez les cellules.

In [ ]:
if RUN_TRAINING:
    # EXERCICE
    # Lancez l'entraînement avec trainer.train().
    # Sauvegardez ensuite le modèle et le tokenizer dans FINAL_MODEL_DIR.
    #
    # Indices :
    # train_result = trainer.train()
    # trainer.save_model(str(FINAL_MODEL_DIR))
    # tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
    raise NotImplementedError("Complétez le lancement et la sauvegarde de l'entraînement.")
else:
    print("Entraînement non lancé.")
    print("Pour entraîner le modèle, mettre RUN_TRAINING = True dans la configuration.")

## 19. Évaluer rapidement le modèle

Si l'entraînement a été lancé, on peut calculer une perte de validation.

La perte seule ne suffit pas à juger un modèle génératif, mais elle donne un premier indice.

In [ ]:
if RUN_TRAINING:
    # EXERCICE
    # Évaluez rapidement le modèle avec trainer.evaluate().
    # Affichez ensuite le résultat.
    raise NotImplementedError("Complétez l'évaluation du modèle.")
else:
    print("Pas d'évaluation : l'entraînement n'a pas été lancé.")

## 20. Fonction de génération

Nous allons maintenant préparer une petite fonction pour générer du texte.

Elle pourra utiliser :

- le modèle de base ;
- le modèle affiné, si l'entraînement a été exécuté et sauvegardé.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=80, temperature=0.9, top_p=0.95):
    # EXERCICE
    # Complétez la fonction de génération.
    #
    # Étapes :
    # 1. Choisir le device : "cuda" si disponible, sinon "cpu".
    # 2. Déplacer le modèle sur ce device.
    # 3. Tokeniser le prompt avec return_tensors="pt".
    # 4. Appeler model.generate(...).
    # 5. Décoder la sortie avec tokenizer.decode(...).
    #
    # Indices :
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # output_ids = model.generate(...)
    # return tokenizer.decode(output_ids[0], skip_special_tokens=False)
    raise NotImplementedError("Complétez la fonction de génération.")

## 21. Générer avec le modèle actuel

Si l'entraînement n'a pas été lancé, cette cellule utilise simplement le modèle de base.

La génération ne sera probablement pas très adaptée au corpus.  
C'est normal : le modèle n'a pas encore appris notre format.

In [ ]:
print("PROMPT :")
print(PROMPT_TEMPLATE)

print("=" * 80)
print("GÉNÉRATION")
print("=" * 80)
print(generate_text(model, tokenizer, PROMPT_TEMPLATE))

## 22. Recharger le modèle affiné, s'il existe

Si un modèle a été sauvegardé dans `FINAL_MODEL_DIR`, on peut le recharger et comparer son comportement avec celui du modèle de base.

In [ ]:
if FINAL_MODEL_DIR.exists():
    print("Chargement du modèle affiné :", FINAL_MODEL_DIR)

    # EXERCICE
    # Rechargez le tokenizer et le modèle affiné depuis FINAL_MODEL_DIR.
    # Puis générez un texte à partir de PROMPT_TEMPLATE.
    #
    # Indices :
    # tuned_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
    # tuned_model = AutoModelForCausalLM.from_pretrained(str(FINAL_MODEL_DIR))
    raise NotImplementedError("Complétez le chargement du modèle affiné.")
else:
    print("Aucun modèle affiné sauvegardé pour le moment.")
    print("Lancez l'entraînement pour créer :", FINAL_MODEL_DIR)